# Autoencoders and Embeddings: Complete Notebook
> **Course:** Machine Learning / Business Intelligence
> **Goal:** Understand learned representations, semantic compression and the latent space

> **Tip:** Turn on the GPU in Colab: `Runtime → Change runtime type → GPU (T4)`

---
## What will you learn?
1. What **embeddings** are and why they are the foundation of modern AI
2. The architecture of an **autoencoder** (encoder → latent space → decoder)
3. Training and evaluating with the **reconstruction loss**
4. Exploring the **latent space**: PCA/t-SNE visualization
5. **Interpolation** between images in the latent space
6. **Vector arithmetic**: semantic analogies
7. **Denoising autoencoder**: learning to remove noise
8. **Anomaly detection** with the reconstruction error

---
### Why do embeddings matter?

| Object | Original space | Embedding | Example |
|---|---|---|---|
| 28×28 image | 784 pixels | 8 dimensions | Digit → coordinate in a semantic space |
| Word | 50,000 tokens | 128-1536 dim. | 'King' - 'Man' + 'Woman' is close to 'Queen' |
| Customer | 200 variables | 16 dimensions | Compressed profile for recommendations |
| Product | 100k catalog | 32 dimensions | Semantic similarity without exhaustive search |

## 1. Installation and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.datasets import fashion_mnist

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cosine, euclidean

plt.rcParams.update({
    'figure.facecolor': '#18181b', 'axes.facecolor': '#27272a',
    'axes.edgecolor': '#3f3f46', 'text.color': '#e4e4e7',
    'axes.labelcolor': '#a1a1aa', 'xtick.color': '#71717a',
    'ytick.color': '#71717a', 'grid.color': '#3f3f46', 'grid.alpha': 0.4,
})

PALETTE = ['#10b981','#3b82f6','#f59e0b','#8b5cf6','#ef4444',
           '#ec4899','#06b6d4','#f97316','#22c55e','#64748b']

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
np.random.seed(42)
tf.random.set_seed(42)

---
## 2. Dataset: Fashion-MNIST

**Fashion-MNIST** replaces the classic MNIST with clothing items. Harder, and more relevant to the real world.

| Class | Item | Class | Item |
|---|---|---|---|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

In [ ]:
(X_train_raw, y_train), (X_test_raw, y_test) = fashion_mnist.load_data()

CLASS_NAMES = ['T-Shirt','Pantalón','Pullover','Vestido','Abrigo',
               'Sandalia','Camisa','Zapatilla','Bolso','Botín']

print(f'Train: {X_train_raw.shape}  |  Test: {X_test_raw.shape}')
print(f'Classes: {np.unique(y_train)}')
print(f'Class distribution: {np.bincount(y_train)} (balanced)')

In [ ]:
# Visualize samples per class
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Fashion-MNIST: 2 samples per class', fontsize=12)
for cls in range(10):
    idxs = np.where(y_train == cls)[0][:2]
    for row, idx in enumerate(idxs):
        ax = axes[row, cls]
        ax.imshow(X_train_raw[idx], cmap='gray', vmin=0, vmax=255)
        if row == 0: ax.set_title(CLASS_NAMES[cls], fontsize=8)
        ax.axis('off')
plt.tight_layout(); plt.show()

---
## 3. Preprocessing

In [ ]:
# Normalize to [0,1] and flatten to 784-dimensional vectors
X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32')  / 255.0

X_train_flat = X_train.reshape(-1, 784)
X_test_flat  = X_test.reshape(-1, 784)

print(f'Shape after preprocessing: {X_train_flat.shape}')
print(f'Value range: [{X_train_flat.min():.2f}, {X_train_flat.max():.2f}]')
print()
print('Difference vs AR/Clustering:')
print('  AR:           transactions -> binary matrix (binarize)')
print('  Clustering:   tabular -> scale (Z-score or Min-Max)')
print('  Autoencoder:  images -> normalize to [0,1], no binarization')

---
## 4. Autoencoder Architecture

```
Input (784)  →  Encoder  →  Latent Space (8)  →  Decoder  →  Output (784)
              784→256→64→8                    8→64→256→784
```

**Bottleneck:** The autoencoder **must learn to compress** the image into just 8 numbers. To reconstruct it well, those 8 numbers have to capture the most important features.

> **Insight:** Nobody told the network which features to extract. The latent space emerges purely from the process of compressing and decompressing.

In [ ]:
LATENT_DIM = 8   # <- latent space dimension (try 2, 4, 8, 16, 32)

# Encoder
encoder_input = keras.Input(shape=(784,), name='encoder_input')
x = layers.Dense(256, activation='relu', name='enc_256')(encoder_input)
x = layers.Dense(64,  activation='relu', name='enc_64')(x)
latent = layers.Dense(LATENT_DIM, activation='linear', name='latent')(x)
encoder = Model(encoder_input, latent, name='encoder')

# Decoder
decoder_input = keras.Input(shape=(LATENT_DIM,), name='decoder_input')
x = layers.Dense(64,  activation='relu', name='dec_64')(decoder_input)
x = layers.Dense(256, activation='relu', name='dec_256')(x)
decoder_output = layers.Dense(784, activation='sigmoid', name='output')(x)
decoder = Model(decoder_input, decoder_output, name='decoder')

# Full autoencoder
ae_input  = keras.Input(shape=(784,))
ae_output = decoder(encoder(ae_input))
autoencoder = Model(ae_input, ae_output, name='autoencoder')
autoencoder.compile(optimizer='adam', loss='mse')

# Summary
print('ENCODER:')
encoder.summary()
print()
print('DECODER:')
decoder.summary()
print()
print(f'Compression ratio: 784 -> {LATENT_DIM}  ({784/LATENT_DIM:.0f}:1)')

---
## 5. Training

In [ ]:
EPOCHS     = 30    # <- try 10, 30, 50
BATCH_SIZE = 256

history = autoencoder.fit(
    X_train_flat, X_train_flat,          # <- input and output are the same image
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_flat, X_test_flat),
    verbose=1,
)

# Loss curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     color='#10b981', linewidth=2, label='Train Loss (MSE)')
ax.plot(history.history['val_loss'], color='#3b82f6', linewidth=2, linestyle='--', label='Val Loss (MSE)')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (reconstruction error)')
ax.set_title('Autoencoder Loss Curves')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

final_train = history.history['loss'][-1]
final_val   = history.history['val_loss'][-1]
print(f'Final train MSE: {final_train:.6f}')
print(f'Final val   MSE: {final_val:.6f}')
if final_val > final_train * 1.5:
    print('Large gap -> possible overfitting, consider reducing epochs or adding dropout')
else:
    print('Train and Val converge well: good generalization')

---
## 6. Reconstruction Quality

In [ ]:
# Reconstruct test images
X_reconstructed = autoencoder.predict(X_test_flat, verbose=0)

# Show original vs reconstructed for each class
n_show = 10
fig, axes = plt.subplots(3, n_show, figsize=(16, 5))
fig.suptitle('Reconstruction per Class: Original | Reconstructed | Error', fontsize=11)

sample_idxs = [np.where(y_test == c)[0][0] for c in range(n_show)]
for col, idx in enumerate(sample_idxs):
    orig  = X_test_flat[idx].reshape(28, 28)
    recon = X_reconstructed[idx].reshape(28, 28)
    err   = np.abs(orig - recon)
    axes[0, col].imshow(orig,  cmap='gray', vmin=0, vmax=1)
    axes[1, col].imshow(recon, cmap='gray', vmin=0, vmax=1)
    axes[2, col].imshow(err,   cmap='Reds', vmin=0, vmax=0.3)
    axes[0, col].set_title(CLASS_NAMES[y_test[idx]], fontsize=7)
    for row in range(3): axes[row, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=8, color='#a1a1aa')
axes[1, 0].set_ylabel('Reconstructed', fontsize=8, color='#a1a1aa')
axes[2, 0].set_ylabel('Error |orig-recon|', fontsize=8, color='#a1a1aa')
for ax_row in [axes[0,0], axes[1,0], axes[2,0]]:
    ax_row.axis('on'); ax_row.set_xticks([]); ax_row.set_yticks([])

plt.tight_layout(); plt.show()

In [ ]:
# Reconstruction error per class
recon_errors = np.mean((X_test_flat - X_reconstructed)**2, axis=1)

fig, ax = plt.subplots(figsize=(11, 4))
for cls in range(10):
    mask  = y_test == cls
    errors_cls = recon_errors[mask]
    ax.boxplot(errors_cls, positions=[cls], widths=0.6,
               patch_artist=True,
               boxprops=dict(facecolor=PALETTE[cls], alpha=0.7),
               medianprops=dict(color='white', linewidth=2),
               whiskerprops=dict(color='#a1a1aa'),
               capprops=dict(color='#a1a1aa'),
               flierprops=dict(marker='.', color=PALETTE[cls], alpha=0.3))

ax.set_xticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Reconstruction MSE')
ax.set_title('Reconstruction Error per Class\n(higher error = class that is harder to compress)')
ax.grid(axis='y')
plt.tight_layout(); plt.show()

---
## 7. Latent Space: Visualization

We will compress 784 dimensions down to 8 (with the encoder) and then to 2 (with PCA or t-SNE) to visualize how the model organizes the clothing items in the latent space.

In [ ]:
# Get embeddings for the test set
N_VIZ = 5000  # subsample for speed
idx_viz = np.random.choice(len(X_test_flat), N_VIZ, replace=False)
X_viz   = X_test_flat[idx_viz]
y_viz   = y_test[idx_viz]

embeddings = encoder.predict(X_viz, verbose=0)   # shape: (N_VIZ, LATENT_DIM)
print(f'Embeddings shape: {embeddings.shape}')
print(f'Each image -> a {LATENT_DIM}-dimensional vector')

# 2D PCA of the latent space
pca2 = PCA(n_components=2, random_state=42)
emb_2d_pca = pca2.fit_transform(embeddings)
print(f'PCA explained variance: {pca2.explained_variance_ratio_.sum():.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Autoencoder Latent Space: Semantic Organization without Supervision', fontsize=12)

# ── PCA ───────────────────────────────────────────────────────────
for cls in range(10):
    mask = y_viz == cls
    axes[0].scatter(emb_2d_pca[mask, 0], emb_2d_pca[mask, 1],
                    c=PALETTE[cls], s=5, alpha=0.6, label=CLASS_NAMES[cls])
axes[0].set_title('2D PCA of the Latent Space (8->2)')
axes[0].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
axes[0].legend(fontsize=7, ncol=2, markerscale=3)

# ── t-SNE (more faithful to local structure) ──────────────────────
print('Computing t-SNE (may take 1-2 minutes)...')
tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=42, verbose=0)
emb_2d_tsne = tsne.fit_transform(embeddings)

for cls in range(10):
    mask = y_viz == cls
    axes[1].scatter(emb_2d_tsne[mask, 0], emb_2d_tsne[mask, 1],
                    c=PALETTE[cls], s=5, alpha=0.6, label=CLASS_NAMES[cls])
axes[1].set_title('2D t-SNE of the Latent Space (8->2)')
axes[1].legend(fontsize=7, ncol=2, markerscale=3)

plt.tight_layout(); plt.show()
print('The autoencoder learned to separate classes without ever seeing the labels.')

---
## 8. Vector Arithmetic in the Latent Space

In a well-organized latent space, **arithmetic operations have semantic meaning**.

The most famous example, with word embeddings:
```
embedding('King') - embedding('Man') + embedding('Woman')  is close to  embedding('Queen')
```

With Fashion-MNIST we can explore:
```
Sneaker - Sandal + Dress  is close to  ?
```

In [ ]:
def class_centroid(cls_id):
    """Average embedding of a class = centroid in the latent space."""
    mask = y_test == cls_id
    return encoder.predict(X_test_flat[mask], verbose=0).mean(axis=0)

centroids = {CLASS_NAMES[i]: class_centroid(i) for i in range(10)}

def vector_arithmetic(a_name, b_name, c_name):
    """Compute A - B + C and find the class closest to the result."""
    result = centroids[a_name] - centroids[b_name] + centroids[c_name]
    # Decoded result image
    result_img = decoder.predict(result.reshape(1, -1), verbose=0)[0].reshape(28, 28)
    # Nearest class by Euclidean distance
    dists = {name: euclidean(result, emb) for name, emb in centroids.items()}
    nearest = sorted(dists.items(), key=lambda x: x[1])
    return result_img, nearest[:3]

# Operations
operations = [
    ('Zapatilla', 'Sandalia', 'Vestido'),
    ('Botín',     'Zapatilla', 'Bolso'),
    ('Abrigo',    'Pullover',  'T-Shirt'),
]

fig, axes = plt.subplots(len(operations), 5, figsize=(12, 7))
fig.suptitle('Vector Arithmetic in the Latent Space\nA - B + C -> ?', fontsize=12)

for row, (a, b, c) in enumerate(operations):
    result_img, nearest = vector_arithmetic(a, b, c)
    a_idx = np.where(y_test == CLASS_NAMES.index(a))[0][0]
    b_idx = np.where(y_test == CLASS_NAMES.index(b))[0][0]
    c_idx = np.where(y_test == CLASS_NAMES.index(c))[0][0]
    for col, (img, title, cmap) in enumerate([
        (X_test_flat[a_idx].reshape(28,28), a, 'gray'),
        (X_test_flat[b_idx].reshape(28,28), f'− {b}', 'gray'),
        (X_test_flat[c_idx].reshape(28,28), f'+ {c}', 'gray'),
        (result_img, '= ?', 'gray'),
        (np.zeros((28,28)), f'→ {nearest[0][0]}', 'gray'),
    ]):
        if col < 4:
            axes[row, col].imshow(img, cmap=cmap, vmin=0, vmax=1)
        else:
            axes[row, col].text(0.5, 0.5, f'Nearest\nclass:\n{nearest[0][0]}\n(d={nearest[0][1]:.2f})',
                                ha='center', va='center', fontsize=8, color='#fbbf24',
                                transform=axes[row, col].transAxes)
        axes[row, col].set_title(title, fontsize=8)
        axes[row, col].axis('off')
    # Visual separators
    for pos, sym in [(1, '−'), (2, '+'), (3, '=')]:
        axes[row, pos].set_title(f'{sym} {operations[row][pos-1] if pos < 3 else "?"}', fontsize=8)

plt.tight_layout(); plt.show()

---
## 9. Interpolation between Images

In [ ]:
def interpolate(img_a_flat, img_b_flat, steps=10):
    """Linear interpolation in the latent space between two images."""
    z_a = encoder.predict(img_a_flat.reshape(1, -1), verbose=0)[0]
    z_b = encoder.predict(img_b_flat.reshape(1, -1), verbose=0)[0]
    alphas = np.linspace(0, 1, steps)
    interpolated = []
    for alpha in alphas:
        z_interp = (1 - alpha) * z_a + alpha * z_b
        img = decoder.predict(z_interp.reshape(1, -1), verbose=0)[0].reshape(28, 28)
        interpolated.append(img)
    return interpolated

# Interesting pairs to interpolate
pairs = [
    (7, 5),   # Sneaker -> Sandal
    (3, 0),   # Dress -> T-Shirt
    (4, 2),   # Coat -> Pullover
]

STEPS = 10
fig, axes = plt.subplots(len(pairs), STEPS, figsize=(16, 5))
fig.suptitle('Interpolation in the Latent Space: Morphing between Clothing Items', fontsize=12)

for row, (cls_a, cls_b) in enumerate(pairs):
    idx_a = np.where(y_test == cls_a)[0][0]
    idx_b = np.where(y_test == cls_b)[0][0]
    frames = interpolate(X_test_flat[idx_a], X_test_flat[idx_b], STEPS)
    for col, img in enumerate(frames):
        axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
        if col == 0:     axes[row, col].set_title(CLASS_NAMES[cls_a], fontsize=7, color='#10b981')
        if col == STEPS-1: axes[row, col].set_title(CLASS_NAMES[cls_b], fontsize=7, color='#3b82f6')

plt.tight_layout(); plt.show()
print('The smooth transition shows that the latent space is continuous and semantically organized.')

---
## 10. Denoising Autoencoder

We train the autoencoder with **noisy images as input** and **clean images as target**. The network learns to filter out the noise.

In [ ]:
# Add Gaussian noise
NOISE_FACTOR = 0.35
X_train_noisy = np.clip(X_train_flat + NOISE_FACTOR * np.random.randn(*X_train_flat.shape), 0, 1)
X_test_noisy  = np.clip(X_test_flat  + NOISE_FACTOR * np.random.randn(*X_test_flat.shape),  0, 1)

# New denoising autoencoder (same architecture)
dae_input  = keras.Input(shape=(784,))
dae_enc    = layers.Dense(256, activation='relu')(dae_input)
dae_enc    = layers.Dense(64,  activation='relu')(dae_enc)
dae_latent = layers.Dense(LATENT_DIM, activation='linear')(dae_enc)
dae_dec    = layers.Dense(64,  activation='relu')(dae_latent)
dae_dec    = layers.Dense(256, activation='relu')(dae_dec)
dae_out    = layers.Dense(784, activation='sigmoid')(dae_dec)
dae = Model(dae_input, dae_out, name='denoising_autoencoder')
dae.compile(optimizer='adam', loss='mse')

# input=noisy, target=clean
dae.fit(X_train_noisy, X_train_flat,
        epochs=20, batch_size=256,
        validation_data=(X_test_noisy, X_test_flat),
        verbose=1)

In [ ]:
# Compare: original | noisy | clean reconstruction
X_denoised = dae.predict(X_test_noisy, verbose=0)

n_show = 10
sample_idxs = [np.where(y_test == c)[0][0] for c in range(n_show)]

fig, axes = plt.subplots(3, n_show, figsize=(16, 5))
fig.suptitle('Denoising Autoencoder: Original | Noisy | Reconstructed', fontsize=11)

for col, idx in enumerate(sample_idxs):
    for row, (img_flat, title) in enumerate([
        (X_test_flat[idx],  CLASS_NAMES[y_test[idx]]),
        (X_test_noisy[idx], '+ noise'),
        (X_denoised[idx],   'denoised'),
    ]):
        axes[row, col].imshow(img_flat.reshape(28,28), cmap='gray', vmin=0, vmax=1)
        if row == 0: axes[row, col].set_title(title, fontsize=7)
        axes[row, col].axis('off')

axes[0,0].set_ylabel('Original',  fontsize=8, color='#a1a1aa')
axes[1,0].set_ylabel('Noisy',     fontsize=8, color='#a1a1aa')
axes[2,0].set_ylabel('Clean',     fontsize=8, color='#a1a1aa')
for r in range(3): axes[r,0].axis('on'); axes[r,0].set_xticks([]); axes[r,0].set_yticks([])
plt.tight_layout(); plt.show()

---
## 11. Anomaly Detection with the Reconstruction Error

**Key idea:** If you train the autoencoder only on normal data, it will have a low reconstruction error on normal data and a **high error on anomalies** (because it never learned to reconstruct them).

We simulate a scenario where the "normal" items are 8 classes and the "anomalies" are 2 unseen classes.

In [ ]:
# Simulate: normal = classes 0-7, anomalies = classes 8 and 9 (Bag and Ankle boot)
NORMAL_CLASSES  = list(range(8))
ANOMALY_CLASSES = [8, 9]

# Train an AE on normal classes only
normal_mask_train = np.isin(y_train, NORMAL_CLASSES)
X_normal = X_train_flat[normal_mask_train]

ae_anomaly_input  = keras.Input(shape=(784,))
x = layers.Dense(256, activation='relu')(ae_anomaly_input)
x = layers.Dense(64,  activation='relu')(x)
x = layers.Dense(LATENT_DIM, activation='linear')(x)
x = layers.Dense(64,  activation='relu')(x)
x = layers.Dense(256, activation='relu')(x)
ae_anomaly_out = layers.Dense(784, activation='sigmoid')(x)
ae_anomaly = Model(ae_anomaly_input, ae_anomaly_out)
ae_anomaly.compile(optimizer='adam', loss='mse')
ae_anomaly.fit(X_normal, X_normal, epochs=15, batch_size=256, verbose=0)

# Compute reconstruction error on the test set
X_recon_test = ae_anomaly.predict(X_test_flat, verbose=0)
recon_err    = np.mean((X_test_flat - X_recon_test)**2, axis=1)

# Error distribution: normal vs anomalies
is_anomaly = np.isin(y_test, ANOMALY_CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Anomaly Detection: Reconstruction Error')

# Histogram
axes[0].hist(recon_err[~is_anomaly], bins=50, color='#10b981', alpha=0.7, label='Normal (classes 0-7)', density=True)
axes[0].hist(recon_err[is_anomaly],  bins=50, color='#ef4444', alpha=0.7, label='Anomaly (classes 8-9)', density=True)
threshold = np.percentile(recon_err[~is_anomaly], 95)
axes[0].axvline(threshold, color='#f59e0b', linewidth=2, linestyle='--', label=f'p95 threshold = {threshold:.4f}')
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Density')
axes[0].set_title('Error Distribution')
axes[0].legend(fontsize=8)

# Boxplot per class
class_errors = [recon_err[y_test == c] for c in range(10)]
bp = axes[1].boxplot(class_errors,
                     patch_artist=True,
                     medianprops=dict(color='white', linewidth=2),
                     whiskerprops=dict(color='#a1a1aa'),
                     capprops=dict(color='#a1a1aa'),
                     flierprops=dict(marker='.', alpha=0.3))
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor('#ef4444' if i >= 8 else '#10b981')
    patch.set_alpha(0.7)
axes[1].set_xticks(range(1, 11))
axes[1].set_xticklabels(CLASS_NAMES, rotation=20, ha='right', fontsize=8)
axes[1].axhline(threshold, color='#f59e0b', linewidth=1.5, linestyle='--')
axes[1].set_title('Error per Class (red = class not seen during training)')
axes[1].grid(axis='y')

plt.tight_layout(); plt.show()

y_pred = (recon_err > threshold).astype(int)
y_true = is_anomaly.astype(int)
from sklearn.metrics import precision_score, recall_score, f1_score
print(f'Threshold = 95th percentile of normal errors: {threshold:.5f}')
print(f'Precision : {precision_score(y_true, y_pred):.3f}')
print(f'Recall    : {recall_score(y_true, y_pred):.3f}')
print(f'F1-Score  : {f1_score(y_true, y_pred):.3f}')

---
## 12. Effect of the Latent Space Dimension

How does the size of the bottleneck affect reconstruction quality?

In [ ]:
def train_ae(latent_dim, epochs=20):
    inp = keras.Input(shape=(784,))
    x   = layers.Dense(256, activation='relu')(inp)
    x   = layers.Dense(64,  activation='relu')(x)
    z   = layers.Dense(latent_dim, activation='linear')(x)
    x   = layers.Dense(64,  activation='relu')(z)
    x   = layers.Dense(256, activation='relu')(x)
    out = layers.Dense(784, activation='sigmoid')(x)
    m   = Model(inp, out)
    m.compile(optimizer='adam', loss='mse')
    h   = m.fit(X_train_flat, X_train_flat, epochs=epochs,
                batch_size=256, validation_split=0.1, verbose=0)
    val_loss = min(h.history['val_loss'])
    return val_loss

latent_dims = [2, 4, 8, 16, 32, 64]
print('Training autoencoders with different latent dimensions...')
val_losses = []
for ld in latent_dims:
    loss = train_ae(ld)
    val_losses.append(loss)
    print(f'  dim={ld:3d}  val_loss={loss:.6f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(latent_dims, val_losses, marker='o', color='#10b981', linewidth=2.5, markersize=8)
ax.fill_between(latent_dims, val_losses, alpha=0.15, color='#10b981')
ax.axvline(8, color='#f59e0b', linewidth=1.5, linestyle='--', label='LATENT_DIM=8 (this notebook)')
ax.set_xlabel('Latent Space Dimension')
ax.set_ylabel('Validation MSE')
ax.set_title('Trade-off: Compression vs Reconstruction Quality')
ax.set_xscale('log', base=2)
ax.set_xticks(latent_dims); ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()
print('More dimensions improve reconstruction, but the embedding loses generalization power.')

---
## 13. Analysis of the Embedding Dimensions

What did each dimension of the latent space "capture"?

In [ ]:
# Get all test-set embeddings
all_embeddings = encoder.predict(X_test_flat, verbose=0)  # (10000, LATENT_DIM)

# For each dimension, vary its value and see what changes in the decoded image
BASE_CLASS = 7  # Sneaker
base_idx   = np.where(y_test == BASE_CLASS)[0][0]
base_emb   = encoder.predict(X_test_flat[[base_idx]], verbose=0)[0]

n_dims = min(LATENT_DIM, 8)
n_vals = 7
fig, axes = plt.subplots(n_dims, n_vals, figsize=(14, n_dims * 1.8))
fig.suptitle(f'Exploring the Latent Dimensions: base image: {CLASS_NAMES[BASE_CLASS]}', fontsize=11)

for dim in range(n_dims):
    dim_range = np.linspace(
        all_embeddings[:, dim].mean() - 2.5 * all_embeddings[:, dim].std(),
        all_embeddings[:, dim].mean() + 2.5 * all_embeddings[:, dim].std(),
        n_vals
    )
    for col, val in enumerate(dim_range):
        z_mod = base_emb.copy()
        z_mod[dim] = val
        img = decoder.predict(z_mod.reshape(1,-1), verbose=0)[0].reshape(28,28)
        axes[dim, col].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[dim, col].axis('off')
    axes[dim, 0].set_ylabel(f'D{dim}', fontsize=8, color='#a1a1aa')

for col, val in enumerate(np.linspace(-2.5, 2.5, n_vals)):
    axes[0, col].set_title(f'{val:.1f}σ', fontsize=7)

plt.tight_layout(); plt.show()
print('Each row controls a different visual aspect: shape, thickness, orientation, texture...')

---
## 14. Exercises

### Exercise 1: Behavior without training
Create a new (untrained) autoencoder and reconstruct 5 images. What do you see? Why is the reconstruction error high? This illustrates that the network has no knowledge before learning.

### Exercise 2: Effect of the architecture
Add a Dropout(0.3) layer after each Dense layer of the encoder. Does it improve or worsen the reconstruction quality and the separation in the latent space?

### Exercise 3: Latent dimension
Train autoencoders with `LATENT_DIM = 2` and `LATENT_DIM = 32`. For `dim=2`, plot the latent space directly (without PCA). Do the classes separate well? What happens to the reconstruction quality?

### Exercise 4: Similarity search
Given any embedding, use sklearn's `NearestNeighbors` to find the 5 most similar images in the latent space. Are they from the same class? This simulates a visual search engine.

```python
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=6).fit(all_embeddings)
distances, indices = nn.kneighbors(query_embedding.reshape(1, -1))
```

### Exercise 5: Connection with LLMs
Text embeddings in models like BERT or GPT have 768-4096 dimensions. What kind of information do you think each dimension captures? How does it differ from the latent space of images?